In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from numpy import mean
from numpy import absolute
from numpy import sqrt
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

In [2]:
#Setting colors for plot
custom_colors =["blue", "sienna", "limegreen", "red", "black",'cyan']
#setting relative path for data
path = "./data"
player_list = ['AB de Villiers','RG Sharma','MS Dhoni','AD Russell','CH Gayle', 'DA Warner', 'V Kohli', 'KD Karthik','S Dhawan', 'SR Watson', 'MP Stoinis']

In [3]:
batting_salary =pd.read_csv(path+"/battingSalaries.csv")
columns = batting_salary.columns.tolist()
cols_to_use = columns[1:len(columns)-1] 
print(columns)
print(len(cols_to_use))

['Player', 'I', 'R', 'B', 'Outs', 'Avg', 'SR', 'HS', '4s', '6s', '50', '100', 'Salary', 'Year']
12


In [4]:
display(batting_salary)

,Player,I,R,B,Outs,Avg,SR,HS,4s,6s,50,100,Salary,Year
0,RD Gaikwad,16,635,466,14,45.36,136.27,101,64,23,4,1,28600,2021
1,F du Plessis,16,634,458,14,45.29,138.43,95,61,23,6,0,250000,2021
2,KL Rahul,13,626,451,10,62.60,138.80,98,48,30,6,0,1718750,2021
3,S Dhawan,16,587,471,15,39.13,124.63,92,63,16,3,0,812500,2021
4,GJ Maxwell,14,513,356,12,42.75,144.10,78,48,21,6,0,2035700,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,Washington Sundar,5,65,38,3,21.67,171.05,35,5,4,0,0,500000,2018
542,WP Saha,10,122,102,8,15.25,119.61,35,17,1,0,0,781250,2018
543,YK Pathan,13,260,200,9,28.89,130.00,45,22,11,0,0,296880,2018
544,YS Chahal,1,0,4,0,0.00,0.00,0,0,0,0,0,937500,2018


In [5]:
batting_salary['Salary'] = batting_salary['Salary'].astype(int)

In [6]:
batting_salary.describe()

,I,R,B,Outs,Avg,SR,HS,4s,6s,50,100,Salary,Year
count,546.000000,546.000000,546.000000,546.000000,546.000000,546.000000,546.000000,546.00000,546.000000,546.000000,546.000000,5.460000e+02,546.000000
mean,6.435897,134.446886,101.326007,5.047619,18.391648,110.320751,35.241758,11.71978,5.608059,0.741758,0.036630,6.262015e+05,2019.507326
std,4.963030,166.885034,119.372810,4.488673,16.520530,49.794822,31.743123,15.99106,7.805917,1.381787,0.206622,6.147735e+05,1.117394
min,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.500000e+04,2018.000000
25%,2.000000,9.250000,11.000000,1.000000,5.000000,87.500000,7.000000,1.00000,0.000000,0.000000,0.000000,1.562500e+05,2019.000000
50%,5.000000,58.500000,46.000000,4.000000,15.125000,119.370000,26.000000,4.00000,2.000000,0.000000,0.000000,4.375000e+05,2019.000000
75%,11.000000,218.500000,161.750000,8.000000,28.750000,139.032500,58.750000,18.00000,9.000000,1.000000,0.000000,9.375000e+05,2021.000000
max,17.000000,735.000000,518.000000,17.000000,101.000000,333.330000,130.000000,68.00000,52.000000,8.000000,2.000000,2.656250e+06,2021.000000


In [7]:
# For batting data only selecting batsmen who satisfy following criteria
# 1) Total score more than 200
# 2) Average score more than 10
# 3) Strike Rate more than 95
data=batting_salary[['Player', 'I', 'R', 'Outs', 'Avg', 'SR', 'HS', '4s', '6s', '50', '100', 'Salary']].groupby(["Player"],as_index=False).agg({'I':'sum', 'R':'sum', 'Outs':'sum', 'Avg':'mean', 'SR':'mean','HS':'mean', '4s':'sum', '6s':'sum', '50':'sum', '100':'sum', 'Salary':'mean'})
data=data[(data['R'] > 200) & (data['Avg'] > 15) & (data['SR'] > 100) ]
print(len(data))
display(data)

67


,Player,I,R,Outs,Avg,SR,HS,4s,6s,50,100,Salary
3,AB de Villiers,52,1689,39,43.5575,158.9100,80.25,126,95,18,0,1718750.0
5,AD Russell,44,1126,36,31.1350,171.6400,61.75,71,106,6,0,1328130.0
7,AJ Finch,21,402,20,19.5400,122.6000,49.00,34,16,1,0,798675.0
10,AM Rahane,36,884,34,20.8325,115.4275,59.50,97,16,3,1,692500.0
14,AT Rayudu,57,1499,45,32.9975,130.1775,75.00,119,70,7,1,343750.0
...,...,...,...,...,...,...,...,...,...,...,...,...
212,V Shankar,33,611,25,27.2950,120.5600,43.50,33,27,2,0,500000.0
214,VR Iyer,10,370,9,41.1100,128.4700,67.00,37,14,4,0,28600.0
216,WP Saha,28,553,25,29.5850,128.8275,48.50,61,12,2,0,323862.5
219,YBK Jaiswal,13,289,13,19.1150,119.5600,42.00,34,12,1,0,342900.0


In [8]:
#Scalling the data
sc_x = StandardScaler()
sc_y = StandardScaler()
x = sc_x.fit_transform(data.iloc[:,1:-1])
y = sc_y.fit_transform(data.iloc[:,-1:])
#print(scaled_data)

## Linear Regression

In [9]:
#define cross-validation method to use
folds = KFold(n_splits = 10, shuffle = False, random_state = 100)

In [10]:
#build multiple linear regression model
from sklearn.feature_selection import RFE
linear_mdl = LinearRegression()
linear_mdl.fit(x,y)
rfe = RFE(linear_mdl)  

In [11]:
#use k-fold CV to evaluate model
#scores = cross_val_score(linear_mdl, x, y, scoring='neg_mean_absolute_error',
 #                        cv=cv, n_jobs=-1)

In [12]:

# set up GridSearchCV()
# step-2: specify range of hyperparameters to tune
hyper_params = [{'n_features_to_select': list(range(1, 11))}]
lm_model_cv = GridSearchCV(estimator = rfe, 
                        param_grid= hyper_params,
                        scoring= 'neg_mean_squared_error', 
                        cv = folds, 
                        verbose = 1,
                        return_train_score=True)      

# fit the model
lm_model_cv.fit(x, y)   

Fitting 10 folds for each of 10 candidates, totalling 100 fits


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\callp\AppData\Local\Continuum\anaconda3\lib

C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-v

C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-v

GridSearchCV(cv=KFold(n_splits=10, random_state=100, shuffle=False),
             error_score='raise-deprecating',
             estimator=RFE(estimator=LinearRegression(copy_X=True,
                                                      fit_intercept=True,
                                                      n_jobs=None,
                                                      normalize=False),
                           n_features_to_select=None, step=1, verbose=0),
             iid='warn', n_jobs=None,
             param_grid=[{'n_features_to_select': [1, 2, 3, 4, 5, 6, 7, 8, 9,
                                                   10]}],
             pre_dispatch='2*n_jobs', refit=True, return_train_score=True,
             scoring='neg_mean_squared_error', verbose=1)

In [13]:
y_pred_ln = sc_y.inverse_transform(lm_model_cv.predict(x))

In [14]:
y_pred_ln

array([1866408.23867915, 1203212.76377359,  435204.53833954,
        446057.99730061, 1220275.03714905,  447202.37708883,
        493629.56221828,  531886.82842784, 1024844.71265102,
        480090.40904298,  524965.9291121 , 1052668.309979  ,
        669898.66294342,  966966.97410938,  394284.43335743,
        632491.32752715, 1085959.52897463,  977981.13440407,
       1201356.29908401, 1105845.34245625,  662004.4358074 ,
        201090.90514877,  487575.19673383, 1236228.29098535,
       1120913.3052502 , 1038062.46315907, 1409729.91312203,
       1266741.6807582 ,  958216.56377637, 1367362.78289552,
        848001.37291645,  957642.32476695, 1366208.05247298,
        781235.37814987,  784489.05896985, 1237360.1659662 ,
        221826.64942369,  522292.73970524,  970070.30423004,
       1135761.63777798,  549657.08847403,  787024.40248477,
        923770.6143582 ,  967380.51178517,  484902.00732697,
       1023126.09385337, 1114301.30379433,  791531.20113952,
        852656.36809745,

In [15]:
df = pd.DataFrame({'Player':data.Player,'Actual': data.Salary, 'Linear Pred': y_pred_ln})
df['Actual'] = df['Actual'].astype(int)
df['Linear Pred'] = df['Linear Pred'].astype(int)
df
display(df[df.Player.isin(player_list)])

,Player,Actual,Linear Pred
3,AB de Villiers,1718750,1866408
5,AD Russell,1328130,1203212
35,CH Gayle,312500,1024844
44,DA Warner,1953130,1052668
94,KD Karthik,1156250,1120913
128,MP Stoinis,827225,957642
130,MS Dhoni,2343750,1366208
164,RG Sharma,2343750,1023126
174,S Dhawan,812500,852656
191,SR Watson,625000,769811


In [16]:
mse_lm= mean_squared_error(data.Salary, y_pred_ln)

### SVR Regression

In [17]:
#building model

In [18]:
svr_mdl = SVR(kernel='rbf', C=10)
svr_mdl.fit(x,y)
#svr_rfe = RFE(svr_mdl) 

C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


SVR(C=10, cache_size=200, coef0=0.0, degree=3, epsilon=0.1,
    gamma='auto_deprecated', kernel='rbf', max_iter=-1, shrinking=True,
    tol=0.001, verbose=False)

In [19]:
#

In [20]:
# fit the model
svr_mdl.fit(x, y)   

C:\Users\callp\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


SVR(C=10, cache_size=200, coef0=0.0, degree=3, epsilon=0.1,
    gamma='auto_deprecated', kernel='rbf', max_iter=-1, shrinking=True,
    tol=0.001, verbose=False)

In [21]:
y_pred_svr = sc_y.inverse_transform(svr_mdl.predict(x))

In [22]:
#player_list = ['AB de Villiers','MS Dhoni','AD Russell', 'V Kohli', 'KD Karthik','Washington Sundar','JJ Bumrah']

In [23]:
#pd.set_option('display.max_rows', None)
#df = pd.DataFrame({'Player':data.Player,'Actual': data.Salary, 'Predicted': y_pred})
#df['Actual'] = df['Actual'].astype(int)
df['SVR Prediction'] = y_pred_svr
df['SVR Prediction'] = df['SVR Prediction'].astype(int)
#display(df)
display(df[df.Player.isin(player_list)])

,Player,Actual,Linear Pred,SVR Prediction
3,AB de Villiers,1718750,1866408,1650879
5,AD Russell,1328130,1203212,1396102
35,CH Gayle,312500,1024844,380726
44,DA Warner,1953130,1052668,1885257
94,KD Karthik,1156250,1120913,1087877
128,MP Stoinis,827225,957642,895369
130,MS Dhoni,2343750,1366208,2275616
164,RG Sharma,2343750,1023126,563981
174,S Dhawan,812500,852656,880649
191,SR Watson,625000,769811,556762


In [24]:
from sklearn.metrics import mean_squared_error
mse_svr= mean_squared_error(data.Salary, y_pred_svr)

In [25]:
print("Mean squared error: %.2f" % mean_squared_error(data.Salary, y_pred_svr))

Mean squared error: 88837274004.12


## Decision Tree Regressoion

In [26]:
from sklearn.tree import DecisionTreeRegressor
dec_tree_regreesion = DecisionTreeRegressor(max_depth=3,random_state=20)

In [27]:
dec_tree_regreesion.fit(x,y)

DecisionTreeRegressor(criterion='mse', max_depth=3, max_features=None,
                      max_leaf_nodes=None, min_impurity_decrease=0.0,
                      min_impurity_split=None, min_samples_leaf=1,
                      min_samples_split=2, min_weight_fraction_leaf=0.0,
                      presort=False, random_state=20, splitter='best')

In [28]:
y_pred_dt = sc_y.inverse_transform(dec_tree_regreesion.predict(x))
y_pred_dt
#print(y_pred_dt.shape)

array([ 944007.31182796,  944007.31182796,  613376.43518519,
        944007.31182796,  944007.31182796,  138407.5       ,
        138407.5       ,  613376.43518519,  944007.31182796,
        613376.43518519,  312564.16666667, 2148440.        ,
        613376.43518519,  613376.43518519,  613376.43518519,
        613376.43518519,  944007.31182796,  944007.31182796,
        944007.31182796,  944007.31182796,  944007.31182796,
        138407.5       ,  613376.43518519,  944007.31182796,
        944007.31182796,  944007.31182796, 2239583.33333333,
        944007.31182796,  944007.31182796,  944007.31182796,
        613376.43518519,  944007.31182796, 2148440.        ,
        613376.43518519,  613376.43518519,  944007.31182796,
        312564.16666667, 2214300.        ,  944007.31182796,
        944007.31182796,  138407.5       ,  613376.43518519,
        944007.31182796,  944007.31182796,  613376.43518519,
        944007.31182796, 2239583.33333333,  944007.31182796,
        812500.        ,

In [29]:
#pd.set_option('display.max_rows', None)
#df = pd.DataFrame({'Player':data.Player,'Actual': data.Salary, 'Predicted': y_pred_dt})
#df['Actual'] = df['Actual'].astype(int)
df['DT Pred'] = y_pred_dt
df['DT Pred'] = df['DT Pred'].astype(int)
#display(df)
display(df[df.Player.isin(player_list)])


,Player,Actual,Linear Pred,SVR Prediction,DT Pred
3,AB de Villiers,1718750,1866408,1650879,944007
5,AD Russell,1328130,1203212,1396102,944007
35,CH Gayle,312500,1024844,380726,944007
44,DA Warner,1953130,1052668,1885257,2148440
94,KD Karthik,1156250,1120913,1087877,944007
128,MP Stoinis,827225,957642,895369,944007
130,MS Dhoni,2343750,1366208,2275616,2148440
164,RG Sharma,2343750,1023126,563981,944007
174,S Dhawan,812500,852656,880649,812500
191,SR Watson,625000,769811,556762,944007


In [30]:
mse_dt= mean_squared_error(data.Salary, y_pred_dt)

In [31]:
# Calculation of Mean Squared Error (MSE)
print("Mean squared error: %.2f" % mean_squared_error(data.Salary, y_pred_dt))

Mean squared error: 206240659846.13


## KNN Regression

In [32]:
from sklearn.neighbors import KNeighborsRegressor

In [33]:
import plotly.graph_objs as go

In [34]:
neighbors = np.arange(1, 20, 2)
scores = []

In [35]:
for k in neighbors:   # running for different K values to know which yields the max accuracy. 
    clf = KNeighborsRegressor(n_neighbors = k,  weights = 'distance', p=1)
    clf.fit(x, y)
    score = cross_val_score(clf, x, y, cv = 10)
    scores.append(score.mean())

In [36]:
mse = [1-x for x in scores]

In [37]:
optimal_k = neighbors[mse.index(min(mse))]
print("Optimal K: ", optimal_k)

Optimal K:  15


In [38]:
# Training the model on Optimal K.
knn_regression = KNeighborsRegressor(n_neighbors = optimal_k)
knn_regression.fit(x, y)

KNeighborsRegressor(algorithm='auto', leaf_size=30, metric='minkowski',
                    metric_params=None, n_jobs=None, n_neighbors=15, p=2,
                    weights='uniform')

In [39]:
y_pred_knn= sc_y.inverse_transform(knn_regression.predict(x))
y_pred_knn

array([[1070834.        ],
       [ 913797.72222222],
       [ 409167.88888889],
       [ 769197.        ],
       [1012500.        ],
       [ 607788.94444444],
       [ 554825.5       ],
       [ 937332.22222222],
       [ 886578.66666667],
       [ 842593.38888889],
       [ 396627.72222222],
       [ 759240.88888889],
       [ 679433.11111111],
       [ 560874.94444444],
       [ 627180.44444444],
       [ 616672.72222222],
       [1094792.        ],
       [ 827610.33333333],
       [ 953995.5       ],
       [1080981.66666667],
       [1009594.22222222],
       [ 620617.38888889],
       [ 792574.22222222],
       [1052048.83333333],
       [ 999632.77777778],
       [ 720479.        ],
       [1053125.33333333],
       [1124057.55555556],
       [ 847648.33333333],
       [1111289.22222222],
       [ 797788.55555556],
       [ 722754.77777778],
       [1181289.22222222],
       [ 535700.        ],
       [ 861519.66666667],
       [ 799731.66666667],
       [ 774539.44444444],
 

In [40]:
df['Knn Pred'] = y_pred_knn
df['Knn Pred'] = df['Knn Pred'].astype(int)
#display(df)
display(df[df.Player.isin(player_list)])

,Player,Actual,Linear Pred,SVR Prediction,DT Pred,Knn Pred
3,AB de Villiers,1718750,1866408,1650879,944007,1070834
5,AD Russell,1328130,1203212,1396102,944007,913797
35,CH Gayle,312500,1024844,380726,944007,886578
44,DA Warner,1953130,1052668,1885257,2148440,759240
94,KD Karthik,1156250,1120913,1087877,944007,999632
128,MP Stoinis,827225,957642,895369,944007,722754
130,MS Dhoni,2343750,1366208,2275616,2148440,1181289
164,RG Sharma,2343750,1023126,563981,944007,799731
174,S Dhawan,812500,852656,880649,812500,948958
191,SR Watson,625000,769811,556762,944007,996818


In [41]:
mse_knn= mean_squared_error(data.Salary, y_pred_knn)

In [42]:
df2 = pd.DataFrame({'Regression Type':np.array(["Linear","SVR","Decision Tree","KNN"]),'MSE Value': np.array([int(mse_lm),int(mse_svr),int(mse_dt),int(mse_knn)])})
df2.sort_values(by=['MSE Value'])


,Regression Type,MSE Value
1,SVR,88837274004
2,Decision Tree,206240659846
0,Linear,340958818595
3,KNN,386682835719


In [43]:
# then end